# 03 公司基本信息数据审计

## 目标

对公司基本信息训练数据 `firm_profile.csv` 进行原始质量审计。

本阶段只识别问题，不直接修改 raw 数据。

重点检查：

- 数据规模与字段结构；
- 股票代码格式；
- 公司名称格式；
- 省份和城市表示；
- 行业字段；
- 所有制字段及缺失值；
- 上市日期格式；
- 公司记录是否重复；
- 与财务面板公司的覆盖关系。

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

assert (PROJECT_ROOT / "pyproject.toml").exists()
assert DATA_RAW.exists()

profile_raw = pd.read_csv(
    DATA_RAW / "firm_profile.csv",
    dtype={
        "stock_code": "string",
        "company_name": "string",
        "province": "string",
        "city": "string",
        "industry": "string",
        "ownership": "string",
    },
)

print("数据读取完成")
print("shape:", profile_raw.shape)

profile_raw.head()

数据读取完成
shape: (42, 7)


,stock_code,company_name,province,city,industry,ownership,listing_date
0,000001,华辰科技有限公司,广东,广州市,软件和信息技术服务业,国有,2005-01-01
1,2,新岳科技股份有限公司,广东省,深圳市,汽车制造业,民营,2006/02/02
2,3,海川科技股份有限公司,北京,北京市,医药制造业,外资,40034
3,000004,中盛科技股份有限公司,北京市,北京市,计算机通信和其他电子设备制造业,国有,NaN
4,000005,宏远科技股份有限公司,浙江,杭州市,专用设备制造业,民营,2009-05-05


In [2]:
profile_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   stock_code    42 non-null     string
 1   company_name  42 non-null     string
 2   province      42 non-null     string
 3   city          42 non-null     string
 4   industry      42 non-null     string
 5   ownership     39 non-null     string
 6   listing_date  32 non-null     str   
dtypes: str(1), string(6)
memory usage: 6.1 KB


In [3]:
profile_raw.dtypes

stock_code      string
company_name    string
province        string
city            string
industry        string
ownership       string
listing_date       str
dtype: object

In [4]:
profile_quality_summary = pd.DataFrame(
    {
        "dtype": profile_raw.dtypes.astype(str),
        "missing": profile_raw.isna().sum(),
        "unique": profile_raw.nunique(dropna=True),
    }
)

profile_quality_summary

,dtype,missing,unique
stock_code,string,0,42
company_name,string,0,42
province,string,0,12
city,string,0,12
industry,string,0,6
ownership,string,3,3
listing_date,str,10,32


In [5]:
print("stock_code dtype:", profile_raw["stock_code"].dtype)

print("\n股票代码长度分布:")
print(
    profile_raw["stock_code"]
    .dropna()
    .str.strip()
    .str.len()
    .value_counts()
    .sort_index()
)

print("\n前 20 个股票代码原始表示:")
print(
    profile_raw["stock_code"]
    .map(repr)
    .head(20)
)

stock_code dtype: string

股票代码长度分布:
stock_code
1     2
2     6
6    34
Name: count, dtype: Int64

前 20 个股票代码原始表示:
0       '000001'
1            '2'
2            '3'
3     ' 000004 '
4       '000005'
5       '000006'
6       '000007'
7       '000008'
8       '000009'
9       '000010'
10      '000011'
11          '12'
12          '13'
13    ' 000014 '
14      '000015'
15      '000016'
16      '000017'
17      '000018'
18      '000019'
19      '000020'
Name: stock_code, dtype: str


In [6]:
stock_code_text = (
    profile_raw["stock_code"]
    .str.strip()
)

invalid_stock_code_mask = (
    stock_code_text.notna()
    & ~stock_code_text.str.fullmatch(r"\d+")
)

print(
    "非纯数字股票代码数量:",
    invalid_stock_code_mask.sum(),
)

print(
    "标准化前唯一股票代码数:",
    profile_raw["stock_code"].nunique(),
)

stock_code_candidate = (
    stock_code_text.str.zfill(6)
)

print(
    "标准化后唯一股票代码数:",
    stock_code_candidate.nunique(),
)

print(
    "标准化后非6位代码数量:",
    (
        stock_code_candidate.notna()
        & ~stock_code_candidate.str.fullmatch(r"\d{6}")
    ).sum(),
)

非纯数字股票代码数量: 0
标准化前唯一股票代码数: 42
标准化后唯一股票代码数: 42
标准化后非6位代码数量: 0


In [7]:
company_name_stripped = (
    profile_raw["company_name"]
    .str.strip()
)

company_name_space_mask = (
    profile_raw["company_name"]
    .ne(company_name_stripped)
    .fillna(False)
)

print(
    "公司名称存在首尾空格的记录数:",
    company_name_space_mask.sum(),
)

pd.DataFrame(
    {
        "before": profile_raw.loc[
            company_name_space_mask,
            "company_name",
        ].map(repr),
        "after": company_name_stripped.loc[
            company_name_space_mask
        ].map(repr),
    }
)

公司名称存在首尾空格的记录数: 3


,before,after
13,' 明德科技股份有限公司 ','明德科技股份有限公司'
26,' 兆丰科技股份有限公司 ','兆丰科技股份有限公司'
39,' 鸿远科技股份有限公司 ','鸿远科技股份有限公司'


In [8]:
print("公司名称后缀统计:")

print(
    pd.Series(
        {
            "股份有限公司": (
                company_name_stripped
                .str.endswith("股份有限公司")
                .sum()
            ),
            "科技有限公司但非股份有限公司": (
                company_name_stripped
                .str.endswith("科技有限公司")
                & ~company_name_stripped
                .str.endswith("股份有限公司")
            ).sum(),
        }
    )
)

公司名称后缀统计:
股份有限公司            38
科技有限公司但非股份有限公司     4
dtype: int64


In [9]:
for col in [
    "province",
    "city",
    "ownership",
]:
    print("=" * 50)
    print(col)

    print(
        profile_raw[col]
        .value_counts(
            dropna=False
        )
        .sort_index()
    )

province
province
上海     3
上海市    3
北京     4
北京市    4
四川省    3
广东     4
广东省    5
江苏     3
江苏省    3
浙江     3
浙江省    4
湖北省    3
Name: count, dtype: int64[pyarrow]
city
city
上海市    6
北京市    8
南京市    3
宁波市    3
广州市    4
成都市    3
杭州市    3
武汉市    3
深圳市    4
珠海市    1
绍兴市    1
苏州市    3
Name: count, dtype: int64[pyarrow]
ownership
ownership
国有      14
外资      12
民营      13
<NA>     3
Name: count, dtype: int64[pyarrow]


In [10]:
listing_text = (
    profile_raw["listing_date"]
    .astype("string")
    .str.strip()
)

listing_dash_mask = (
    listing_text.str.fullmatch(
        r"\d{4}-\d{2}-\d{2}",
        na=False,
    )
)

listing_slash_mask = (
    listing_text.str.fullmatch(
        r"\d{4}/\d{2}/\d{2}",
        na=False,
    )
)

listing_excel_serial_mask = (
    listing_text.str.fullmatch(
        r"\d+",
        na=False,
    )
)

listing_missing_mask = (
    listing_text.isna()
)

listing_unknown_mask = ~(
    listing_dash_mask
    | listing_slash_mask
    | listing_excel_serial_mask
    | listing_missing_mask
)

print(
    "YYYY-MM-DD:",
    listing_dash_mask.sum(),
)

print(
    "YYYY/MM/DD:",
    listing_slash_mask.sum(),
)

print(
    "Excel 序列号形式:",
    listing_excel_serial_mask.sum(),
)

print(
    "缺失:",
    listing_missing_mask.sum(),
)

print(
    "未知格式:",
    listing_unknown_mask.sum(),
)

YYYY-MM-DD: 11
YYYY/MM/DD: 11
Excel 序列号形式: 10
缺失: 10
未知格式: 0


In [11]:
profile_raw.loc[
    listing_excel_serial_mask,
    [
        "stock_code",
        "company_name",
        "listing_date",
    ],
]

,stock_code,company_name,listing_date
2,3,海川科技股份有限公司,40034
6,000007,瑞华科技股份有限公司,40102
10,000011,东泰科技股份有限公司,40170
14,000015,永盛科技股份有限公司,40238
18,000019,鼎新科技股份有限公司,40306
22,23,联创科技有限公司,40374
26,000027,兆丰科技股份有限公司,40442
30,000031,东方科技股份有限公司,40510
34,000035,科创科技股份有限公司,40578
38,000039,中信科技股份有限公司,40646


In [12]:
province_map = {
    "上海": "上海市",
    "上海市": "上海市",
    "北京": "北京市",
    "北京市": "北京市",
    "广东": "广东省",
    "广东省": "广东省",
    "江苏": "江苏省",
    "江苏省": "江苏省",
    "浙江": "浙江省",
    "浙江省": "浙江省",
    "四川省": "四川省",
    "湖北省": "湖北省",
}

province_text = (
    profile_raw["province"]
    .str.strip()
)

unmapped_province_mask = (
    province_text.notna()
    & ~province_text.isin(province_map)
)

print(
    "无法按当前规则映射的省份记录数:",
    unmapped_province_mask.sum(),
)

province_candidate = (
    province_text.map(province_map)
)

print(
    "标准化前省份类别数:",
    province_text.nunique(),
)

print(
    "标准化后省份类别数:",
    province_candidate.nunique(),
)

province_candidate.value_counts().sort_index()

无法按当前规则映射的省份记录数: 0
标准化前省份类别数: 12
标准化后省份类别数: 7


province
上海市    6
北京市    8
四川省    3
广东省    9
江苏省    6
浙江省    7
湖北省    3
Name: count, dtype: int64

In [13]:
city_text = (
    profile_raw["city"]
    .str.strip()
)

print(
    "城市首尾空格记录数:",
    (
        profile_raw["city"]
        .ne(city_text)
        .fillna(False)
    ).sum(),
)

print(
    "不以“市”结尾的城市记录数:",
    (
        city_text.notna()
        & ~city_text.str.endswith("市")
    ).sum(),
)

print(
    "城市类别数:",
    city_text.nunique(),
)

print(
    city_text
    .value_counts()
    .sort_index()
)

城市首尾空格记录数: 0
不以“市”结尾的城市记录数: 0
城市类别数: 12
city
上海市    6
北京市    8
南京市    3
宁波市    3
广州市    4
成都市    3
杭州市    3
武汉市    3
深圳市    4
珠海市    1
绍兴市    1
苏州市    3
Name: count, dtype: int64[pyarrow]


In [14]:
ownership_text = (
    profile_raw["ownership"]
    .astype("string")
    .str.strip()
)

print(
    "ownership 缺失数:",
    ownership_text.isna().sum(),
)

print(
    "ownership 非缺失类别数:",
    ownership_text.nunique(dropna=True),
)

print(
    ownership_text
    .value_counts(dropna=False)
    .sort_index()
)

ownership 缺失数: 3
ownership 非缺失类别数: 3
ownership
国有      14
外资      12
民营      13
<NA>     3
Name: count, dtype: int64[pyarrow]


In [15]:
ownership_pseudo_missing = (
    ownership_text
    .fillna("")
    .isin(
        [
            "",
            "-",
            "--",
            "NA",
            "N/A",
            "未知",
        ]
    )
    & ownership_text.notna()
)

print(
    "ownership 伪缺失记录数:",
    ownership_pseudo_missing.sum(),
)

ownership 伪缺失记录数: 0


In [16]:
listing_date_candidate = pd.Series(
    pd.NaT,
    index=profile_raw.index,
    dtype="datetime64[ns]",
)

listing_date_candidate.loc[
    listing_dash_mask
] = pd.to_datetime(
    listing_text.loc[
        listing_dash_mask
    ],
    format="%Y-%m-%d",
)

listing_date_candidate.loc[
    listing_slash_mask
] = pd.to_datetime(
    listing_text.loc[
        listing_slash_mask
    ],
    format="%Y/%m/%d",
)

excel_serial_values = pd.to_numeric(
    listing_text.loc[
        listing_excel_serial_mask
    ],
    errors="raise",
)

listing_date_candidate.loc[
    listing_excel_serial_mask
] = pd.to_datetime(
    excel_serial_values,
    unit="D",
    origin="1899-12-30",
)

In [17]:
excel_date_check = pd.DataFrame(
    {
        "original": listing_text.loc[
            listing_excel_serial_mask
        ],
        "converted": listing_date_candidate.loc[
            listing_excel_serial_mask
        ],
    }
)

excel_date_check

,original,converted
2,40034,2009-08-09
6,40102,2009-10-16
10,40170,2009-12-23
14,40238,2010-03-01
18,40306,2010-05-08
22,40374,2010-07-15
26,40442,2010-09-21
30,40510,2010-11-28
34,40578,2011-02-04
38,40646,2011-04-13


In [18]:
print(
    "原始非缺失上市日期:",
    listing_text.notna().sum(),
)

print(
    "成功转换日期:",
    listing_date_candidate.notna().sum(),
)

print(
    "转换后缺失日期:",
    listing_date_candidate.isna().sum(),
)

print(
    "最早上市日期:",
    listing_date_candidate.min(),
)

print(
    "最晚上市日期:",
    listing_date_candidate.max(),
)

原始非缺失上市日期: 32
成功转换日期: 32
转换后缺失日期: 10
最早上市日期: 2005-01-01 00:00:00
最晚上市日期: 2019-09-20 00:00:00


In [19]:
financials_clean = pd.read_parquet(
    DATA_RAW.parent
    / "processed"
    / "firm_financials_clean.parquet"
)

financial_codes = set(
    financials_clean[
        "stock_code"
    ].astype("string")
)

profile_codes = set(
    stock_code_candidate
)

print(
    "财务面板公司数:",
    len(financial_codes),
)

print(
    "Profile 公司数:",
    len(profile_codes),
)

print(
    "财务公司在 Profile 中缺失:",
    len(
        financial_codes
        - profile_codes
    ),
)

print(
    "Profile 中财务样本外公司:",
    len(
        profile_codes
        - financial_codes
    ),
)

财务面板公司数: 40
Profile 公司数: 42
财务公司在 Profile 中缺失: 0
Profile 中财务样本外公司: 2


In [20]:
missing_from_profile = sorted(
    financial_codes
    - profile_codes
)

extra_in_profile = sorted(
    profile_codes
    - financial_codes
)

print(
    "财务中有、Profile 中没有:",
    missing_from_profile,
)

print(
    "Profile 中有、财务中没有:",
    extra_in_profile,
)

财务中有、Profile 中没有: []
Profile 中有、财务中没有: ['900001', '900002']


## 审计结论

本 Notebook 对公司基本信息原始数据进行了第一轮质量审计。

主要发现如下：

- 原始数据共 42 条公司级记录、7 个字段；
- `stock_code` 无缺失，共 42 个唯一值；
- 部分股票代码存在前导零丢失或首尾空格，但均为纯数字；
- 股票代码候选标准化为 6 位后仍保持 42 个唯一值，不发生代码碰撞；
- `company_name` 无缺失，其中 3 条存在首尾空格；
- 公司名称存在“股份有限公司”和“有限公司”等不同合法后缀，不进行机械统一；
- `province` 共出现 12 种原始表示，主要由省级地区简称和正式名称混用造成，标准化后对应 7 个省级地区；
- `city` 共 12 类，无首尾空格，全部以“市”结尾；
- `industry` 共 6 类，无缺失；
- `ownership` 共 3 个非缺失类别，同时存在 3 条真实缺失，无伪缺失；
- `listing_date` 存在 `YYYY-MM-DD`、`YYYY/MM/DD` 和 Excel 日期序列号三种有效格式；
- 32 条非缺失上市日期均可成功转换，另外 10 条为原始缺失；
- 转换后上市日期范围为 2005-01-01 至 2019-09-20；
- 公司基本信息能够覆盖财务面板全部 40 家公司；
- Profile 额外包含股票代码 `900001` 和 `900002` 两家公司。

本阶段只进行审计和候选转换验证，不修改 raw 数据。

下一阶段将在独立 Notebook 中制定并执行正式清洗规则。